# 模拟题 B — Trees: Salary Prediction（进阶版）

**数据集：** `employee_salary.csv`

**变量说明：**
- `employee_id`: 员工 ID（删除）
- `hire_year`: 入职年份 → 创建 `tenure = 2023 - hire_year`（工龄）
- `education`: 学历等级（1–4）
- `department_size`: 部门人数
- `performance_score`: 绩效评分（1–5）
- `overtime_hours`: 月均加班时长
- `promotions`: 晋升次数
- `remote_work`: 是否远程办公（1=是，0=否）
- `certifications`: 持有证书数量
- `manager`: 是否为管理层（1=是，0=否）
- `city_tier`: 城市级别（1=一线，2=二线，3=三线）
- `salary`: 年薪（美元）【目标变量】

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

def rmse(y_true, y_pred):
    return round(np.sqrt(mean_squared_error(y_true, y_pred)), 2)

In [ ]:
# B1 ─────────────────────────────────────────────────────────────────
# 读入数据，创建 tenure = 2023 - hire_year，删除 employee_id 和 hire_year。
# 用 pd.qcut(salary, q=100) 分层，80/20 分割，random_state=99。
# (a) train 中 tenure 的平均值？
# (b) train 和 test 中 salary 均值是否接近？（验证分层有效性）

# data = pd.read_csv('employee_salary.csv')
# data['tenure'] = 2023 - data['hire_year']
# y = data['salary']
# X = data.drop(columns=['salary', 'hire_year', 'employee_id'])
# y_binned = pd.qcut(data['salary'], q=100)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, train_size=0.8, random_state=99, stratify=y_binned)
# print('Avg tenure (train):', round(X_train['tenure'].mean(), 2))
# print('Train salary mean:', round(y_train.mean(), 2))
# print('Test  salary mean:', round(y_test.mean(), 2))

In [ ]:
# B2 ─────────────────────────────────────────────────────────────────
# 计算所有特征与 salary 的绝对相关系数。
# (a) 最强相关的特征是什么？
# (b) 如果用相关系数选变量，会遗漏哪些信息？

# train_full = X_train.copy()
# train_full['salary'] = y_train.values
# corr = train_full.corr()['salary'].drop('salary').abs().sort_values(ascending=False)
# print(corr)

# 📌 遗漏的信息：
# 相关系数只衡量线性双变量关系。树模型能捕捉非线性效应和交互作用，
# 低相关变量在树中可能仍作为有效分裂节点出现。

In [ ]:
# B3 ─────────────────────────────────────────────────────────────────
# 用 tenure 拟合回归树，max_depth=2，可视化树。
# (a) 树的第一个分裂节点是什么条件？
# (b) 预测工龄 1 年的员工薪资？
# (c) 预测工龄 20 年的员工薪资？

# tree3 = DecisionTreeRegressor(max_depth=2, random_state=99)
# tree3.fit(X_train[['tenure']], y_train)
# plt.figure(figsize=(10,4))
# plot_tree(tree3, feature_names=['tenure'], filled=True, rounded=True)
# plt.show()
# print(tree3.predict(pd.DataFrame({'tenure': [1, 20]})))

In [ ]:
# B4 ─────────────────────────────────────────────────────────────────
# 分别构建三棵树（全变量）：depth=2, depth=10, maximal（无约束）。
# 填写下表，分析 Bias-Variance Tradeoff：
#
#  模型           Train RMSE   Test RMSE   过拟合/欠拟合？
#  depth=2        ？           ？          ？
#  depth=10       ？           ？          ？
#  maximal        ？           ？          ？

# for depth in [2, 10, None]:
#     tree = DecisionTreeRegressor(max_depth=depth, random_state=99)
#     tree.fit(X_train, y_train)
#     tr = rmse(y_train, tree.predict(X_train))
#     te = rmse(y_test,  tree.predict(X_test))
#     label = f'depth={depth}' if depth else 'maximal'
#     print(f'{label:12s}: Train={tr:>10,}  Test={te:>10,}')

In [ ]:
# B5 ─────────────────────────────────────────────────────────────────
# 用 GridSearchCV（cv=5）在 max_depth=[2,3,5,8,12,15] 上做网格搜索。
# (a) 最优 max_depth 是多少？
# (b) 打印所有候选 depth 的 CV RMSE，观察随 depth 增加的趋势。
# (c) 为什么 depth 过大的模型 CV RMSE 反而变差？

# gs = GridSearchCV(
#     DecisionTreeRegressor(random_state=99),
#     {'max_depth': [2, 3, 5, 8, 12, 15]},
#     cv=5, scoring='neg_root_mean_squared_error')
# gs.fit(X_train, y_train)
# print('Best depth:', gs.best_params_)
# for p, s in zip(gs.cv_results_['params'], gs.cv_results_['mean_test_score']):
#     print(f"  depth={p['max_depth']}: CV RMSE={round(-s,2)}")

In [ ]:
# B6 ─────────────────────────────────────────────────────────────────
# 多参数网格搜索（cv=5），寻找最优组合：
#   max_depth:        [3, 8, 9, 12]
#   min_samples_leaf: [5, 20, 50, 100]
#   max_features:     [2, 4, 6, 'sqrt']
#
# (a) 最优参数组合是什么？
# (b) Tuned 模型的 Train RMSE 和 Test RMSE？
# (c) 与 depth=2 和 maximal 模型相比，Test RMSE 改善了多少？

# param_grid = {
#     'max_depth':        [3, 8, 9, 12],
#     'min_samples_leaf': [5, 20, 50, 100],
#     'max_features':     [2, 4, 6, 'sqrt']
# }
# gs2 = GridSearchCV(
#     DecisionTreeRegressor(random_state=99),
#     param_grid, cv=5,
#     scoring='neg_root_mean_squared_error')
# gs2.fit(X_train, y_train)
# best = gs2.best_estimator_
# print('Best params:', gs2.best_params_)
# print('Tuned Train RMSE:', rmse(y_train, best.predict(X_train)))
# print('Tuned Test  RMSE:', rmse(y_test,  best.predict(X_test)))

In [ ]:
# B7 ─────────────────────────────────────────────────────────────────
# 【代码填空题】以下代码有 3 处错误，找出并改正。

# 错误代码：
# y_binned = pd.qcut(data['salary'], q=100)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     train_size=0.3,            # 错误 1
#     random_state=99,
#     stratify=y_binned)
#
# tree = DecisionTreeRegressor(max_depth=5)
# tree.fit(X_test, y_test)       # 错误 2
#
# test_rmse = rmse(y_train, tree.predict(X_train))   # 错误 3

# 正确代码：
# 错误1：train_size=0.3 → 应为 0.7（或 0.8），题目要求 70%（或 80%）在 train
# 错误2：tree.fit(X_test, y_test) → 应为 tree.fit(X_train, y_train)，模型应用训练集拟合
# 错误3：rmse(y_train, tree.predict(X_train)) → 应为 rmse(y_test, tree.predict(X_test))，计算 test RMSE

In [ ]:
# B8 ─────────────────────────────────────────────────────────────────
# 【概念题】请解释以下现象：
# 某同学构建了一棵 maximal tree，发现 Train RMSE = 0，但 Test RMSE = 80,000。
# (a) 这说明了什么问题？
# (b) 如何解决？
# (c) 为什么 cv=5 的 GridSearchCV 能帮助找到更好的模型？

# 答案框架：
# (a) 严重过拟合（high variance）：
#     Maximal tree 完全记忆了训练数据的每一个样本（Train RMSE=0），
#     但没有泛化能力，在新数据上误差极大。
#
# (b) 解决方法：
#     - 限制 max_depth（控制树的深度）
#     - 增大 min_samples_leaf（叶节点最少样本数）
#     - 限制 max_features（每次分裂时随机考虑的特征数）
#     - 使用 GridSearchCV + CV 找最优超参数
#
# (c) 5-fold CV 的原理：
#     将训练集分为 5 份，轮流用其中 4 份训练、1 份验证，
#     取 5 次验证误差的平均值作为该超参数组合的评估分数。
#     这样可以在不接触 test 集的情况下估计泛化误差，
#     从而选择 bias-variance 平衡最好的超参数。

In [ ]:
# B9 ─────────────────────────────────────────────────────────────────
# 【综合题】请解释 pd.qcut(price, q=100) 在分层抽样中的作用。
# 如果不做分层，可能出现什么问题？

# 答案框架：
# pd.qcut(price, q=100) 将 price 分成 100 个等频区间作为分层变量。
# 作用：确保 train 和 test 集中 price 的分布（高价房、低价房比例）保持一致。
#
# 不分层的风险：
# 随机分割可能导致 train 中高价房比例偏高、test 中低价房比例偏高（或反之），
# 使得 test RMSE 不能准确反映模型的真实泛化能力。
# 尤其在 price 分布偏态（长尾）时，这个问题更严重。